In [1]:
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Load final XGBoost test predictions
# ------------------------------------------------------------

df = pd.read_csv(
    "../data/xgboost_test_predictions.csv",
    parse_dates=["observation_date"]
)

print(df.shape)
df.head()

(11645, 10)


,account_id,observation_date,is_churn,current_plan,current_mrr,upgrade_last_90d,downsell_last_90d,xgb_churn_probability,risk_rank,monthly_revenue_at_risk
0,1967,2020-04-13,0,plus,19.0,0,0,0.491160,1.0,9.332045
1,11348,2020-04-27,0,basic,10.0,0,0,0.476819,2.0,4.768186
2,12030,2020-05-01,1,basic,10.0,0,0,0.460800,3.0,4.608003
3,3709,2020-04-11,0,premium,35.0,0,0,0.454001,4.0,15.890023
4,995,2020-05-01,0,basic,9.0,0,0,0.390919,5.0,3.518268


In [2]:
# ------------------------------------------------------------
# Top 10% highest-risk observations
# ------------------------------------------------------------

cutoff = df["xgb_churn_probability"].quantile(0.90)

df["retention_priority"] = np.where(
    df["xgb_churn_probability"] >= cutoff,
    "High Priority",
    "Standard"
)

priority_summary = (
    df.groupby("retention_priority")["is_churn"]
      .agg(
          observations="count",
          churners="sum",
          churn_rate="mean"
      )
)

priority_summary

,observations,churners,churn_rate
retention_priority,,,
High Priority,1165,60,0.051502
Standard,10480,128,0.012214


In [3]:
top10 = df[df["retention_priority"] == "High Priority"]

total_churners = df["is_churn"].sum()
captured_churners = top10["is_churn"].sum()

capture_rate = captured_churners / total_churners

overall_churn_rate = df["is_churn"].mean()
top10_churn_rate = top10["is_churn"].mean()

lift = top10_churn_rate / overall_churn_rate

print("Customers targeted:", len(top10))
print("Churners identified:", captured_churners)
print("Total churners:", total_churners)
print(f"Churners captured: {capture_rate:.2%}")
print(f"Overall churn rate: {overall_churn_rate:.2%}")
print(f"Top 10% churn rate: {top10_churn_rate:.2%}")
print(f"Lift: {lift:.2f}x")

Customers targeted: 1165
Churners identified: 60
Total churners: 188
Churners captured: 31.91%
Overall churn rate: 1.61%
Top 10% churn rate: 5.15%
Lift: 3.19x


In [5]:
# Revenue at risk

df["expected_monthly_revenue_at_risk"] = (
    df["xgb_churn_probability"]
    * df["current_mrr"]
)

high_risk = (
    df[df["retention_priority"] == "High Priority"]
    .sort_values(
        "expected_monthly_revenue_at_risk",
        ascending=False
    )
)

high_risk[
    [
        "account_id",
        "observation_date",
        "current_plan",
        "current_mrr",
        "xgb_churn_probability",
        "expected_monthly_revenue_at_risk",
        "upgrade_last_90d",
        "downsell_last_90d"
    ]
].head(20)

,account_id,observation_date,current_plan,current_mrr,xgb_churn_probability,expected_monthly_revenue_at_risk,upgrade_last_90d,downsell_last_90d
3,3709,2020-04-11,premium,35.00,0.454001,15.890023,0,0
0,1967,2020-04-13,plus,19.00,0.491160,9.332045,0,0
21,1379,2020-05-02,premium,33.25,0.248891,8.275614,0,0
32,2919,2020-04-23,premium,35.00,0.230248,8.058681,0,0
5,10836,2020-04-15,plus,20.00,0.377166,7.543323,0,0
38,7009,2020-04-24,premium,35.00,0.214518,7.508115,0,0
9,2566,2020-04-17,plus,19.00,0.357454,6.791617,0,0
24,10427,2020-04-11,plus,20.00,0.240065,4.801291,0,0
1,11348,2020-04-27,basic,10.00,0.476819,4.768186,0,0
23,6778,2020-04-18,plus,19.00,0.242704,4.611385,0,0


In [6]:
print(
    "Expected monthly revenue at risk across test observations:",
    round(df["expected_monthly_revenue_at_risk"].sum(), 2)
)

print(
    "Expected monthly revenue at risk in High Priority group:",
    round(high_risk["expected_monthly_revenue_at_risk"].sum(), 2)
)

Expected monthly revenue at risk across test observations: 3036.17
Expected monthly revenue at risk in High Priority group: 1200.78


In [7]:
high_risk.to_csv(
    "../data/high_priority_retention_customers.csv",
    index=False
)

In [8]:
print("Test rows:", len(df))
print("Unique accounts:", df["account_id"].nunique())

Test rows: 11645
Unique accounts: 11645
